# Cross-Encoder Hallucination Detection

Fine‑tuning **DeBERTa‑v3‑base** to classify grounded vs. hallucinated RAG responses.

<small>Author: *Tahlee Stone* · Last updated: 10 July 2025 </small>

## Overview
This notebook walks through:
1. **Data preprocessing** – HaluBench + domain‑specific (retail bank customer service) synthetic dataset.
2. **Preprocessing & tokenisation** – chunk long contexts, encode with DeBERTa.
3. **Model training** Fine‑tune using `Trainer` API (PyTorch backend)
4. **Model evaluation** – validation metrics, test‑set classification report.
5. **Saving artefacts** for downstream analysis or deployment.



In [ ]:
# Import required packages and clear caching
import gc
gc.collect()
import torch
torch.cuda.empty_cache()
import pandas as pd
import random, numpy as np
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from typing import List, Dict
from torch.nn.functional import softmax
import itertools

SEED = 42
set_seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

## 1  Data Preprocessing

In [ ]:
# Load the dataset
df = pd.read_parquet("hf://datasets/PatronusAI/HaluBench/data/test-00000-of-00001.parquet")
df.to_csv("halubench_local.csv", index=False)
#df = pd.read_csv("halubench_local_raw.csv")

# Exclude HaluEval and COVIDQA
gen_df = df[~df['source_ds'].isin(['halueval', 'covidQA'])].copy()
print(f"Total observations in gen_df (excluding 'halueval'): {len(gen_df)}")

# Map labels and rename columns
label_map = {'PASS': 1, 'FAIL': 0}
gen_df['label'] = gen_df['label'].map(label_map)
gen_df = gen_df.rename(columns={'question': 'query','passage': 'context','answer': 'response',})
gen_df['source'] = 'halubench'

# HaluBench: Stratified split 70/15/15
train, temp = train_test_split(gen_df, test_size=0.3, stratify=gen_df['label'],random_state=42)
val, test = train_test_split(temp, test_size=0.5, stratify=temp['label'], random_state=42)
display(gen_df.head())

# Save All Split
train.to_csv("gen_train.csv", index=False)
val.to_csv("gen_val.csv", index=False)
test.to_csv("gen_test.csv", index=False)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Total observations in gen_df (excluding 'halueval'): 3900


,id,context,query,response,label,source_ds,source
0,d3fb4c3c-d21b-480a-baa0-98d6d0d17c1d,Hoping to rebound from the road loss to the Ch...,Which team scored the longest field goal kick ...,"['Rams', 'second', 'Marc Bulger', 'Kevin Curtis']",0,DROP,halubench
1,8603663e-c53b-46db-a482-a867f12ff3b4,"As of the census of 2000, there were 218,590 p...",How many percent were not Irish?,87.1,0,DROP,halubench
2,c63a73e5-2c91-489b-bd24-af150ddfa82c,Hoping to rebound from the road loss to the Ch...,How many yards was the second longest field go...,42,0,DROP,halubench
3,52db14ed-5426-46ec-b0ae-4ef843b2d692,Hoping to rebound from their tough overtime ro...,How long was the last touchdown?,18-yard,0,DROP,halubench
4,31b36417-aad1-412c-b0e5-9c1faaed233f,"As of the census of 2000, there were 218,590 p...",How many in percent from the census weren't Ir...,87.1,0,DROP,halubench


In [ ]:
# Load and Split Domain-Specific Dataset (Paired Q-C-R triplets)
ds_df = pd.read_csv("banking_77_1k_triplet_set_long.csv")

# Add missing columns to match HaluBench
ds_df['source_ds'] = 'bank_qcr'
ds_df['source'] = 'bank_qcr'
ds_df['id'] = ['bankqcr_' + str(i) for i in range(len(ds_df))]

label_map = {'PASS': 1, 'FAIL': 0}
ds_df['label'] = ds_df['label'].map(label_map)

# Assumes each grounded/ungrounded pair shares the same query + context
ds_df['pair_id'] = ds_df['query'] + '||' + ds_df['context']

# Reorder columns to match HaluBench
ds_df = ds_df[['id', 'context', 'query', 'response', 'label', 'source_ds', 'source', 'pair_id']]
display(ds_df.head())

# Get unique pairs (e.g., one for each pair of grounded/ungrounded)
unique_pairs = ds_df['pair_id'].unique()

# Split on pair level to avoid leakage
train_ids, temp_ids = train_test_split(unique_pairs, test_size=0.3, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

# Create splits
train_dom = ds_df[ds_df['pair_id'].isin(train_ids)].reset_index(drop=True)
val_dom  = ds_df[ds_df['pair_id'].isin(val_ids)].reset_index(drop=True)
test_dom = ds_df[ds_df['pair_id'].isin(test_ids)].reset_index(drop=True)

# Drop pair_id after split if no longer needed
for df_ in [train_dom, val_dom, test_dom]:
    df_.drop(columns=["pair_id"], inplace=True)

# Save down split files
train_dom.to_csv("ds_train.csv", index=False)
val_dom.to_csv("ds_val.csv", index=False)
test_dom.to_csv("ds_test.csv", index=False)

# === Final Summary ===
print("HaluBench (General) Split Sizes:")
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

print("Domain-Specific (Bank QCR) Split Sizes:")
print(f"Train_DS: {len(train_dom)}, Val_DS: {len(val_dom)}, Test_DS: {len(test_dom)}")

,id,context,query,response,label,source_ds,source,pair_id
0,bankqcr_0,### How to Edit Your Personal Details\n\nKeepi...,How do I edit my details?,"To edit your personal details, please follow t...",1,bank_qcr,bank_qcr,How do I edit my details?||### How to Edit You...
1,bankqcr_1,"### Cash Withdrawal Charges\n\nAt [Bank Name],...",is there a charge on withdrawals?,When you withdraw cash from an ATM within our ...,1,bank_qcr,bank_qcr,is there a charge on withdrawals?||### Cash Wi...
2,bankqcr_2,**Understanding Transfer Fees**\n\nAt [Bank Na...,Why am I seeing a fee for transferring money?,The fee you are seeing for transferring money ...,1,bank_qcr,bank_qcr,Why am I seeing a fee for transferring money?|...
3,bankqcr_3,### Troubleshooting Declined Card Payments\n\n...,"I keep trying to make a payment, but it doesn'...",I'm sorry to hear you're having trouble with y...,1,bank_qcr,bank_qcr,"I keep trying to make a payment, but it doesn'..."
4,bankqcr_4,**Children's Accounts: An Overview**\n\nAt [Yo...,Do you have a children account available?,"Yes, we offer a Children’s Account designed fo...",1,bank_qcr,bank_qcr,Do you have a children account available?||**C...


HaluBench (General) Split Sizes:
Train: 2730, Val: 585, Test: 585
Domain-Specific (Bank QCR) Split Sizes:
Train_DS: 1400, Val_DS: 300, Test_DS: 300


## 2  Tokenise pairs with DeBERTa tokenizer

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
import pandas as pd

model_name = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def chunk_and_tokenize(df: pd.DataFrame, max_length=512, stride=128):
    cls_token_id = tokenizer.cls_token_id
    sep_token_id = tokenizer.sep_token_id
    all_chunks = []

    for idx, row in df.iterrows():
        query = row['query']
        response = row['response']
        context = row['context']
        label = row['label']  # assumes label column exists

        # Tokenize query + response
        query_ids = tokenizer.encode(query, add_special_tokens=False)
        response_ids = tokenizer.encode(response, add_special_tokens=False)
        qr_ids = [cls_token_id] + query_ids + [sep_token_id] + response_ids + [sep_token_id]
        qr_len = len(qr_ids)

        # Tokenize context
        context_ids = tokenizer.encode(context, add_special_tokens=False)
        available_len = max_length - qr_len - 1  # reserve space for final [SEP]

        if available_len <= 0:
            continue  # skip long QR pairs

        for i in range(0, len(context_ids), available_len - stride):
            context_chunk = context_ids[i:i + available_len]
            input_ids = qr_ids + context_chunk + [sep_token_id]
            attention_mask = [1] * len(input_ids)

            all_chunks.append({
                'input_ids': input_ids,
                'attention_mask': attention_mask,
                'labels': label,
                'chunk_id': f"{idx}_{i}"
            })

    return Dataset.from_list(all_chunks)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [ ]:
# Concatenate the two datasets (HaluBench and Domain)
train_combined = pd.concat([train, train_dom], ignore_index=True)
val_combined = pd.concat([val, val_dom], ignore_index=True)
test_combined = pd.concat([test, test_dom], ignore_index=True)

# Then chunk by context length
train_ds = chunk_and_tokenize(train_combined)
val_ds = chunk_and_tokenize(val_combined)
test_ds = chunk_and_tokenize(test_combined)

In [ ]:
# Check the number of examples created
print(f"Total training examples: {len(train_combined)}")
print(f"Total chunked training examples: {len(train_ds)}")
# View one example
print(train_ds [0])

#  Check length of input_ids (should be ≤ 512)
lengths = [len(x['input_ids']) for x in train_ds]
print(f"Min length: {min(lengths)}")
print(f"Max length: {max(lengths)}")

# Manually decode to verify the structure
decoded = tokenizer.decode(train_ds[0]['input_ids'], skip_special_tokens=False)
print(decoded)

#Check that labels are intact after chunking:
from collections import Counter
print(Counter(train_ds['labels']))

# Extract original index from chunk_id
df_chunks = pd.DataFrame(train_ds)
df_chunks['original_idx'] = df_chunks['chunk_id'].apply(lambda x: int(x.split('_')[0]))

# Count how many chunks per original row
chunk_counts = df_chunks['original_idx'].value_counts()
print(chunk_counts.describe())

Total training examples: 4130
Total chunked training examples: 7051
{'input_ids': [1, 44029, 272, 274, 281, 266, 613, 20135, 7343, 260, 10519, 262, 776, 900, 293, 3576, 478, 439, 272, 269, 1569, 267, 262, 916, 974, 1502, 1548, 294, 339, 269, 13325, 974, 1276, 280, 268, 475, 395, 1210, 265, 751, 265, 3244, 1758, 283, 266, 4639, 265, 2606, 292, 15189, 6661, 264, 15189, 66347, 302, 10519, 267, 2339, 265, 864, 268, 263, 1489, 264, 311, 26385, 470, 260, 2, 1259, 260, 765, 440, 2, 64275, 578, 260, 67886, 82520, 430, 2400, 38407, 16273, 63670, 23265, 30518, 13325, 974, 1276, 66944, 121932, 8988, 31011, 82520, 430, 2487, 112302, 287, 547, 3543, 261, 2488, 605, 752, 3909, 285, 2292, 2026, 1374, 1947, 261, 16789, 5070, 1112, 12437, 67868, 6615, 419, 706, 261, 23686, 419, 989, 261, 50174, 419, 621, 261, 38402, 4613, 1698, 456, 261, 50482, 453, 261, 55372, 453, 261, 38085, 6090, 2161, 7566, 602, 261, 44987, 940, 261, 45520, 1154, 261, 22029, 12437, 48790, 6938, 265, 4027, 404, 261, 21199, 404, 261

## 3  Define evaluation metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

## 6  Fine‑tune DeBERTa‑v3‑base with classification head

In [ ]:
# Assess different hyperparameters and save per-run results
param_grid = {
    'learning_rate': [1e-5, 2e-5, 5e-5],
    'per_device_train_batch_size': [4, 8],
    'num_train_epochs': [3, 5],
}

results = []

for lr, bs, epochs in itertools.product(
    param_grid['learning_rate'],
    param_grid['per_device_train_batch_size'],
    param_grid['num_train_epochs']
):
    print(f"Running run: lr={lr}, batch_size={bs}, epochs={epochs}")

    run_name = f"lr{lr}_bs{bs}_ep{epochs}"

    training_args = TrainingArguments(
        output_dir =f"./results/{run_name}",
        eval_strategy='epoch',
        save_strategy='epoch',
        learning_rate=lr,
        per_device_train_batch_size=bs,
        per_device_eval_batch_size=bs,
        num_train_epochs=epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model='f1',
        logging_steps=20,
        report_to='none'
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        'microsoft/deberta-v3-base',
        num_labels=2
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )

    train_output = trainer.train()
    eval_metrics = trainer.evaluate()

    train_metrics = train_output.metrics

    run_record = {
        "run_name": run_name,
        "learning_rate": lr,
        "batch_size": bs,
        "epochs": epochs,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"eval_{k}": v for k, v in eval_metrics.items()},
    }

    results.append(run_record)

df_results = pd.DataFrame(results)
print("\n=== Grid Search Results ===")
print(df_results)                          # Show table in-notebook
df_results.to_csv('grid_search_results.csv', index=False)

Running run: lr=1e-05, batch_size=4, epochs=3


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-11-3852530546.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.458200,0.520851,0.753272,0.779564,0.785880,0.782709
2,0.222200,0.882279,0.734293,0.760227,0.774306,0.767202
3,0.377600,0.826709,0.728403,0.734587,0.813657,0.772103


...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.701200,0.692201,0.565445,0.565445,1.000000,0.722408
2,0.633400,0.642786,0.669503,0.706559,0.710648,0.708598
3,0.599800,0.610435,0.660340,0.777778,0.559028,0.650505


Running run: lr=5e-05, batch_size=8, epochs=5


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-11-3852530546.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.692200,0.690022,0.565445,0.565445,1.000000,0.722408
2,0.695300,0.684782,0.565445,0.565445,1.000000,0.722408
3,0.694500,0.684767,0.565445,0.565445,1.000000,0.722408
4,0.692300,0.684524,0.565445,0.565445,1.000000,0.722408
5,0.698000,0.684440,0.565445,0.565445,1.000000,0.722408



=== Grid Search Results ===
           run_name  learning_rate  batch_size  epochs  train_train_runtime  \
0   lr1e-05_bs4_ep3        0.00001           4       3            2992.7975   
1   lr1e-05_bs4_ep5        0.00001           4       5            5032.9835   
2   lr1e-05_bs8_ep3        0.00001           8       3            2996.7780   
3   lr1e-05_bs8_ep5        0.00001           8       5            5259.1047   
4   lr2e-05_bs4_ep3        0.00002           4       3            3231.8714   
5   lr2e-05_bs4_ep5        0.00002           4       5            5585.5822   
6   lr2e-05_bs8_ep3        0.00002           8       3            3116.9200   
7   lr2e-05_bs8_ep5        0.00002           8       5            5402.1353   
8   lr5e-05_bs4_ep3        0.00005           4       3            3328.1534   
9   lr5e-05_bs4_ep5        0.00005           4       5            5230.6797   
10  lr5e-05_bs8_ep3        0.00005           8       3            3218.5933   
11  lr5e-05_bs8_ep5    